# Лекција 01 - Увод у AI агенте

Добродошли у прву лекцију курса **AI агенти за почетнике**!

**AI агент** је програм који користи велики језички модел (LLM) као свој мотор за размишљање и може предузимати *радње* у стварном свету — позивати API-је, упитивати базе података или извршавати код — како би остварио циљ у име корисника.

У овом нотебооку ћете направити свог првог агента: **Туристичког агента** који препоручује дестинације за одмор. Током овог процеса научићете како да:

1. Повежете се на Azure AI Foundry Agent Service користећи **Microsoft Agent Framework**.
2. Дате агенту **алат** — једну обичну Python функцију коју агент може да позове.
3. Покренете агента и погледате његов одговор.
4. Стримујете одговор агента токен по токен.


## Подешавање

Пре покретања овог бележника, уверите се да имате:

1. **Azure AI Foundry пројекат** са постављеним моделом за ћаскање (нпр. `gpt-4o-mini`).
2. **Пријављени сте преко Azure CLI** — покрените `az login` у вашем терминалу.
3. **Постављене потребне променљиве окружења:**
   - `AZURE_AI_PROJECT_ENDPOINT` — крајња тачка вашег Azure AI Foundry пројекта.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег постављеног модела.

Испод ћелије инсталира Python пакете који су вам потребни.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Креирање Вашег Првог Агента

Агенту су потребне две ствари:

- **Упутства** која му говоре *ко је* и *како да се понаша* (системски упит).
- **Алати** — Python функције украшене са `@tool` које агент може да позове да би добио информације или извршио радње.

Испод дефинишемо једноставан алат који враћа списак популарних дестинација за одмор. Агент ће користити овај алат када корисник затражи препоруке за путовања.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Стреаминг одговора

За интерактивније искуство можете **стреамовати** одговор агента. Уместо да чекате цео одговор, агент испоручује делове текста како се генеришу. Ово је нарочито корисно у интерфејсима за ћаскање где желите да приказујете резултате у реалном времену.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

У овој лекцији сте научили како да:

- **Креирате провајдера** који се повезује на Azure AI Foundry Agent Service преко `AzureAIProjectAgentProvider`.
- **Дефинишете алатку** користећи `@tool` декоратор како би агент могао да позива ваше Python функције.
- **Покренете агента** са корисничком поруком и одштампате његов одговор.
- **Стримујете одговоре** за излаз у реалном времену.

У следећој лекцији ћемо детаљније истражити агентске оквире и научити како агенти добијају моћније алатке и могућности за вишестепено разматрање.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање одговорности**:
Овај документ је преведен коришћењем AI услуге за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, молимо имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Изворни документ на његовом матичном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
